# E2 — synthetic Kaggle capacity and recovery check

This notebook generates **non-PE synthetic bytes only**. It does not fetch RanDS,
read S3 credentials, upload samples, or start a scientific training run.
Use a private notebook and a reviewed source bundle containing the repository,
`uv.lock`, `.python-version` and `configs/`; exclude `.env`, datasets and credentials.
Provide uv **0.11.9** on PATH before execution. Enable one T4; the second GPU is unused.

The default uses the LMLM E2 profile (detached context, embedding padding_idx=None)
and a maximum 1,048,576-token synthetic input.
The fixture uses batch 64, accumulation 1, three planned epochs and 20 bootstrap draws:
these are **feasibility settings**, not the 15-epoch E2 result. All 128 train inputs
reach the token limit and contain distinct deterministic random bytes.
Both invocations pause before test evaluation. CPU tests independently compare
resumed and continuous model/optimizer/scheduler/RNG state exactly.


In [ ]:
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys

REPO = Path(os.environ.get("MALWEAVE_REPO", "/kaggle/working/malweave")).resolve()
WORK = Path(os.environ.get("E2_SMOKE_ROOT", "/kaggle/working/e2-smoke")).resolve()
# To test a new Kaggle session, attach the previous private notebook output and
# set this to its e2-smoke directory. Source/config/lock must be identical.
RESTORE_FROM = os.environ.get("E2_SMOKE_RESTORE_FROM", "")
DEVICE = os.environ.get("E2_SMOKE_DEVICE", "cuda:0")
MAX_TOKENS = int(os.environ.get("E2_SMOKE_MAX_TOKENS", "1048576"))
SESSION_SECONDS = 1800
assert (REPO / "uv.lock").is_file(), "Supply the reviewed source bundle first."
assert not (REPO / ".env").exists(), "Do not include workstation secrets in the source bundle."
version = subprocess.check_output(["uv", "--version"], text=True)
assert version.startswith("uv 0.11.9 "), version
# Keep the dependency environment out of saved notebook outputs.
ENV = dict(os.environ, UV_PROJECT_ENVIRONMENT="/tmp/malweave-e2-env")
if RESTORE_FROM:
    restore = Path(RESTORE_FROM)
    run_root = restore / "runs" / "synthetic-e2"
    for required in (restore / "config.yaml", restore / "split.csv", restore / "raw",
                     run_root / "config.yaml", run_root / "run-manifest.json",
                     run_root / "last.pt", run_root / "model.safetensors"):
        assert required.exists(), f"Incomplete recovery output: {required}"
subprocess.run(["uv", "sync", "--locked", "--no-default-groups", "--group", "train"],
               cwd=REPO, env=ENV, check=True)
UV = ["uv", "run", "--locked", "--no-default-groups", "--group", "train"]

def run(*args):
    subprocess.run([*UV, *args], cwd=REPO, env=ENV, check=True)


Inspect the runtime before benchmarking. T4 does not need BF16 or Mamba kernels
for this FP32 MalConvGCT check. GPU memory is per device, not pooled.


In [ ]:
run("python", "-c", "import json, torch; print(json.dumps({'torch': str(torch.__version__), 'cuda': torch.version.cuda, 'devices': [{'name': torch.cuda.get_device_name(i), 'memory_bytes': torch.cuda.get_device_properties(i).total_memory} for i in range(torch.cuda.device_count())]}, indent=2))")
if RESTORE_FROM:
    assert not WORK.exists(), "Restore into a new working directory."
    shutil.copytree(Path(RESTORE_FROM), WORK)
else:
    run("malweave", "experiment", "prepare-e2-smoke", "--output-dir", str(WORK),
        "--max-tokens", str(MAX_TOKENS), "--full-length")

COMMON = ["malweave", "experiment", "train", "--track", "malconvgct",
          "--experiment", str(WORK / "config.yaml"),
          "--split-manifest", str(WORK / "split.csv"),
          "--raw-root", str(WORK / "raw"), "--exe-root", str(WORK / "UNUSED_EXE"),
          "--artifact-root", str(WORK / "runs"), "--run-id", "synthetic-e2",
          "--device", DEVICE, "--gradient-accumulation-steps", "1", "--seed", "0",
          "--max-seconds", str(SESSION_SECONDS)]
RUN = WORK / "runs" / "synthetic-e2"
if not RESTORE_FROM:
    run(*COMMON, "--max-updates", "2")
BEFORE = json.loads((RUN / "benchmark.json").read_text())
assert BEFORE["test_evaluated"] is False
manifest = json.loads((RUN / "run-manifest.json").read_text())
assert manifest["batch_size"] == 64
assert manifest["gradient_accumulation_steps"] == 1
behavior = manifest["malconv_source_behavior"]
assert behavior["implementation"] == "lmlm"
assert behavior["context_receives_gradients"] is False
assert behavior["embedding_padding_idx"] is None
assert not (RUN / "test-predictions.csv").exists()
print(json.dumps(BEFORE, indent=2))


The second CLI invocation is a new process. It reloads the complete training
state and commits one additional optimizer update. Stop at the first cell above
if you want to save private output and exercise this step in a new Kaggle session.
Do not publish the real experiment's private manifests or checkpoints.


In [ ]:
target = BEFORE["updates"] + 1
run(*COMMON, "--resume", "--max-updates", str(target))
AFTER = json.loads((RUN / "benchmark.json").read_text())
assert AFTER["updates"] == target
assert AFTER["session_samples"] == 64
assert AFTER["test_evaluated"] is False
assert not (RUN / "test-predictions.csv").exists()
result = {
    "synthetic_only": True,
    "malconv_source_behavior": behavior,
    "recovery_committed_next_update": True,
    "before_updates": BEFORE["updates"],
    "after_updates": AFTER["updates"],
    "session_samples": AFTER["session_samples"],
    "session_seconds": AFTER["session_seconds"],
    "samples_per_second": AFTER["session_samples"] / AFTER["session_seconds"],
    "peak_memory_bytes": AFTER["peak_memory_bytes"],
    "test_evaluated": False,
}
(WORK / "recovery-check.json").write_text(json.dumps(result, indent=2) + "\n")
print(json.dumps(result, indent=2))


Follow progress in `e2-smoke/runs/synthetic-e2/progress.jsonl`.
Save the **private notebook output** after the deliberate pause, and verify the
files are available before ending the session. The output contains the generated
synthetic input, config, audit and `last.pt`, so it can be attached for restoration.
Only load your own trusted recovery state. Kaggle's session disk alone is not a backup.

These measurements establish synthetic capacity/recovery only. They do not predict
RanDS throughput or accuracy. Before a real E2 run, complete the data-handling review,
verify the round-02 split/summary, resolve E2 input audit collisions, benchmark real
train examples, and establish durable checkpoint delivery across session loss.
Do not silently change precision, token limits, cohort, or epoch budget.
